In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json
# --- Filter and prepare PA_data as before ---
PA_data = pd.read_csv('/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/RoentGen-v2/real_data_not_summarized/clean_all_summarized_PA_data.csv')
PA_data['final_impression'] = PA_data['summarized'].fillna(PA_data['impression'])
PA_data = PA_data.drop(columns=['input_ids', 'attention_mask', 'n_input_tokens', 'truncated'])
PA_data['sentence_after_summary'] = PA_data.apply(lambda row: f"{int(row['anchor_age'])} year old {row['ethnicity']} {row['gender']}. {row['final_impression']}", axis=1)


In [47]:
all_summarized_PA_data.columns

Index(['original_index', 'index', 'study', 'impression', 'findings',
       'last_paragraph', 'comparison', 'study_id', 'dicom_id', 'subject_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'gender', 'anchor_age',
       'ethnicity', 'age_group', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
       'image', 'folder', 'impression_length', 'sentence', 'stratify_key',
       'summarized_x', 'final_impression', 'sentence_after_summary',
       'input_ids', 'attention_mask', 'n_input_tokens', 'truncated', 'id',
       'custom_id', 'summarized_y', 'impression_id', 'final_final'],
      dtype='objec

long impression -> summarized --> final_impression

check sentence (final_impression + anchor_age + gender + ethnicity), then summarize the impression of the long sentences

# Tokenizer

In [ ]:
# Tokenize impressions in PA_data using Stable Diffusion's tokenizer
from transformers import AutoTokenizer
token = 'hf_TOKEN_REMOVED'
# Choose the tokenizer consistent with training code (CLIP tokenizer from SD v1-4)
model_id = "stanfordmimi/RoentGen-v2"
try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        subfolder="tokenizer",
        use_fast=True,
        trust_remote_code=True,
        token=token,
    )
except Exception:
    # Fallback to default loading if subfolder is not available
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True, token=token)


# Prepare texts (handle NaNs)
texts = PA_data['impression'].fillna("").astype(str)

# Get tokenizer max length (defaults to 77 for CLIP)
if hasattr(tokenizer, 'model_max_length'):
    max_len = tokenizer.model_max_length
else:
    max_len = 77  # fallback CLIP default

# Batch tokenize for efficiency (with truncation and max_length)
encodings = tokenizer(
    texts.tolist(),
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_attention_mask=True,
)

# Check for entries that exceeded the token limit BEFORE truncation (by re-tokenizing with no truncation)
lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False)) for t in texts]
PA_data['length_impressions'] = lengths



# Prepare texts (handle NaNs)
texts = PA_data['final_impression'].fillna("").astype(str)

# Get tokenizer max length (defaults to 77 for CLIP)
if hasattr(tokenizer, 'model_max_length'):
    max_len = tokenizer.model_max_length
else:
    max_len = 77  # fallback CLIP default

# Batch tokenize for efficiency (with truncation and max_length)
encodings = tokenizer(
    texts.tolist(),
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_attention_mask=True,
)

# Check for entries that exceeded the token limit BEFORE truncation (by re-tokenizing with no truncation)
lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False)) for t in texts]
PA_data['length_final_impressions'] = lengths

# Prepare texts (handle NaNs)
texts = PA_data['final_impression'].fillna("").astype(str)

# Get tokenizer max length (defaults to 77 for CLIP)
if hasattr(tokenizer, 'model_max_length'):
    max_len = tokenizer.model_max_length
else:
    max_len = 77  # fallback CLIP default

# Batch tokenize for efficiency (with truncation and max_length)
encodings = tokenizer(
    texts.tolist(),
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_attention_mask=True,
)

# Check for entries that exceeded the token limit BEFORE truncation (by re-tokenizing with no truncation)
lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False)) for t in texts]
PA_data['length_final_impressions'] = lengths




/home/vito/ibrahimm/.conda/envs/roentgen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (79 > 77). Running this sequence through the model will result in indexing errors


Number of samples truncated due to token limit (77): 1207
                                sentence_after_summary  n_input_tokens
18   68 year old Black female. Heart size is normal...              79
62   66 year old White female. In comparison with t...              81
142  79 year old White male. 1. PICC line tip proba...              79
170  65 year old White female. In comparison with t...              80
184  58 year old White male. No change in appearanc...              78
Tokenizer vocab size: 49408
Max length used: 77
First input_ids length: 77


In [3]:
long_impressions = PA_data[PA_data['truncated']==True]
long_impressions =long_impressions.reset_index(drop=False).rename(columns={'level_0': 'index_in_original'})

In [23]:
mapping_df = pd.read_csv('long_impressionid_customid_mapping_3.csv')
summarized_batch3 = summarized_batch3.merge(mapping_df, on='custom_id',  how='left')


In [36]:
PA_data = PA_data.reset_index(drop=False).rename(columns={'level_0': 'original_index'})

In [37]:
all_summarized_PA_data = PA_data.merge(summarized_batch3, left_on='original_index', right_on='impression_id', how='left')

In [48]:
all_summarized_PA_data[~all_summarized_PA_data['summarized_y'].isna()]

,original_index,index,study,impression,findings,last_paragraph,comparison,study_id,dicom_id,subject_id,...,sentence_after_summary,input_ids,attention_mask,n_input_tokens,truncated,id,custom_id,summarized_y,impression_id,final_final
18,18,77,s59984865,Heart size is normal. This ascending aorta is...,NaN,NaN,___,59984865,2d8a8525-19d0c810-045e1619-ef196132-cd4f1710,10001884,...,68 year old Black female. Heart size is normal...,"[49406, 277, 279, 935, 896, 1449, 3970, 269, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",79,True,batch_req_6917778591c881908c1cb470218b7e78,request-1,Normal heart size; ascending aorta stable. Lun...,18.0,Normal heart size; ascending aorta stable. Lun...
62,62,350,s50208870,"In comparison with the study of ___, there is ...",NaN,NaN,NaN,50208870,14afc73f-777de566-c8ff4345-ac179f43-9eb831de,10006431,...,66 year old White female. In comparison with t...,"[49406, 277, 277, 935, 896, 1579, 3970, 269, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",81,True,batch_req_691777859888819088a8c640b5b2de56,request-2,Little change vs prior. Cardiac silhouette nor...,62.0,Little change vs prior. Cardiac silhouette nor...
142,142,809,s52153377,1. PICC line tip probably lies beyond the SVC/...,The left IJ central line has been removed. The...,NaN,Chest x-ray from earlier the same day.,52153377,6bc14657-810b05e0-4bd32106-c30afa91-77f0122c,10018081,...,79 year old White male. 1. PICC line tip proba...,"[49406, 278, 280, 935, 896, 1579, 2801, 269, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",79,True,batch_req_69177785b2b481908541b635b39cc8e8,request-3,PICC tip likely beyond SVC/RA junction; retrac...,142.0,PICC tip likely beyond SVC/RA junction; retrac...
170,170,1016,s59713053,"In comparison with the study of ___, there is ...",NaN,NaN,NaN,59713053,bb24b5d4-2bbe7463-a188e737-24e07be1-5c31258d,10021927,...,65 year old White female. In comparison with t...,"[49406, 277, 276, 935, 896, 1579, 3970, 269, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",80,True,batch_req_691777858e34819082cbe27600005715,request-4,"Compared to prior, elevated right hemidiaphrag...",170.0,"Compared to prior, elevated right hemidiaphrag..."
184,184,1092,s54088844,No change in appearance as compared to the pre...,NaN,NaN,___,54088844,3e530690-9880e823-e166b246-02faa260-897a251e,10024120,...,58 year old White male. No change in appearanc...,"[49406, 276, 279, 935, 896, 1579, 2801, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",78,True,batch_req_6917778791108190823d3892a6d0de46,request-5,Stable exam. 2 mm calcified right upper lobe g...,184.0,Stable exam. 2 mm calcified right upper lobe g...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70118,70118,375390,s56068595,Faint predominantly linear opacities in the ri...,PA and lateral views of the chest ___ at 18 49...,NaN,Comparison to plain film study dated ___ at 22:45,56068595,e76f3cc4-86287cb9-f1291269-1d6e711d-071ddf39,19957675,...,72 year old White male. Faint predominantly li...,"[49406, 278, 273, 935, 896, 1579, 2801, 269, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",82,True,batch_req_69177832d93c8190a1fb54eb3f183bbb,request-1203,Faint linear opacities in RUL and both lower l...,70118.0,Faint linear opacities in RUL and both lower l...
70186,70186,375808,s55773224,"In comparison to ___ chest radiograph, increas...",NaN,NaN,NaN,55773224,a89be236-3b417aa6-3f7e02d8-010f7a44-f876c4f6,19966756,...,72 year old Black male. In comparison to ___ c...,"[49406, 278, 273, 935, 896, 1449, 2801, 269, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",80,True,batch_req_69177832ffdc8190b509a71b0158a3bd,request-1204,"Compared to prior, cardiomegaly increased; wor...",70186.0,"Compared to prior, cardiomegaly increased; wor..."
70187,70187,375810,s56263589,"Compared to chest radiographs since ___, most ...",NaN,NaN,NaN,56263589,aae29fc9-8082062a-70406a14-4548e3ee-4b0f2c73,19966756,...,72 year old

In [39]:
all_summarized_PA_data['final_final'] = all_summarized_PA_data['summarized_y'].fillna(all_summarized_PA_data['final_impression'])


In [41]:
all_summarized_PA_data.to_csv('all_summarized_PA_data.csv', index=False)

In [45]:
all_summarized_PA_data.columns

Index(['original_index', 'index', 'study', 'impression', 'findings',
       'last_paragraph', 'comparison', 'study_id', 'dicom_id', 'subject_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'gender', 'anchor_age',
       'ethnicity', 'age_group', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
       'image', 'folder', 'impression_length', 'sentence', 'stratify_key',
       'summarized_x', 'final_impression', 'sentence_after_summary',
       'input_ids', 'attention_mask', 'n_input_tokens', 'truncated', 'id',
       'custom_id', 'summarized_y', 'impression_id', 'final_final'],
      dtype='objec

In [46]:
all_summarized_PA_data

,original_index,index,study,impression,findings,last_paragraph,comparison,study_id,dicom_id,subject_id,...,sentence_after_summary,input_ids,attention_mask,n_input_tokens,truncated,id,custom_id,summarized_y,impression_id,final_final
0,0,0,s53189527,No acute cardiopulmonary abnormality.,"The cardiac, mediastinal and hilar contours ar...",NaN,___,53189527,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,...,52 year old White female. No acute cardiopulmo...,"[49406, 276, 273, 935, 896, 1579, 3970, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",19,False,NaN,NaN,NaN,NaN,No acute cardiopulmonary abnormality.
1,1,3,s50414267,No acute cardiopulmonary process.,"There is no focal consolidation, pleural effus...",NaN,None.,50414267,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,...,52 year old White female. No acute cardiopulmo...,"[49406, 276, 273, 935, 896, 1579, 3970, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",17,False,NaN,NaN,NaN,NaN,No acute cardiopulmonary process.
2,2,22,s55697293,Stable chest radiograph.,Heart size is normal. Mediastinal contours ar...,NaN,"PA and lateral chest x-ray, ___.",55697293,c50494f1-90e2bff5-e9189550-1a4562fd-6ab5204c,10000935,...,52 year old Black female. Stable chest radiogr...,"[49406, 276, 273, 935, 896, 1449, 3970, 269, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ...",14,False,NaN,NaN,NaN,NaN,Stable chest radiograph.
3,3,25,s50985099,"Compared to chest radiographs since ___, most ...",NaN,NaN,NaN,50985099,6ad03ed1-97ee17ee-9cf8b320-f7011003-cd93b42d,10000980,...,73 year old Black female. Compared to prior CX...,"[49406, 278, 274, 935, 896, 1449, 3970, 269, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",55,False,NaN,NaN,NaN,NaN,Compared to prior CXRs: edema and possible pne...
4,4,27,s54935705,Mild pulmonary edema with superimposed left up...,There is mild pulmonary edema with superimpose...,NaN,___.,54935705,6ad819bb-bae74eb9-7b663e90-b8deabd7-57f8054a,10000980,...,73 year old Black female. Mild pulmonary edema...,"[49406, 278, 274, 935, 896, 1449, 3970, 269, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",32,False,NaN,NaN,NaN,NaN,Mild pulmonary edema with superimposed left up...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70428,70428,376998,s52434977,No acute cardiopulmonary abnormality.,"There is no focal consolidation, pleural effus...",NaN,"Multiple priors, most recently on ___.",52434977,4750f069-a4a2b152-61dadb2b-8e7c09a6-1c0578c2,19999068,...,63 year old White male. No acute cardiopulmona...,"[49406, 277, 274, 935, 896, 1579, 2801, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",19,False,NaN,NaN,NaN,NaN,No acute cardiopulmonary abnormality.
70429,70429,377000,s50847545,No acute cardiopulmonary process.,Frontal and lateral views of the chest. No pr...,NaN,NaN,50847545,8dc9f5e1-14887015-8db378ef-2fd4441a-d45ee0f3,19999156,...,62 year old White female. No acute cardiopulmo...,"[49406, 277, 273, 935, 896, 1579, 3970, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",17,False,NaN,NaN,NaN,NaN,No acute cardiopulmonary process.
70430,70430,377011,s53282218,PA and lateral chest compared to a series of c...,NaN,NaN,NaN,53282218,5a5eddf4-b64e5e49-f6e9c8bc-d6409b00-015470ea,19999287,...,71 year old Black female. CXR: ~22 mm opacity ...,"[49406, 278, 272, 935, 896, 1449, 3970, 269, 6...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,False,NaN,NaN,NaN,NaN,CXR: ~22 mm opacity over cardiac apex (possibl...
70431,70431,377018,s57132437,No acute cardiothoracic process.,"The lungs are clear, and the cardiomediastinal...",NaN,None.,57132437,3fcd0406-9b111603-feae7033-96632b3a-111333e5,19999733,...,19 year old White female. No acute cardiothora...,"[49406, 272, 280, 935, 896, 1579, 3970, 269, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",17,False,NaN,NaN,NaN,NaN,No acute cardiothoracic process.


# openai

## A. prepare

In [51]:
import json

# Prepare a batch jsonl file for OpenAI /v1/chat/completions endpoint and create mapping between impression id and custom id
batch_jsonl_filename = "long_impressions_openai_chat_batch_3.jsonl"
impression_customid_mapping_filename = "long_impressionid_customid_mapping_3.csv"
system_message = "You are a helpful assistant."
user_task = 'Your task is to summarize this radiology report within 200 characters or less. Your response must be concise, truthful, and keep all relevant medical information.'
model_name = "gpt-5"  # Or set to your desired model

impressions_df = long_impressions  # include index (impression id)
custom_id_list = []
impression_id_list = []

with open(batch_jsonl_filename, "w", encoding="utf-8") as fout:
    for idx, row in impressions_df.iterrows():
        imp = row['final_impression']
        impression_id = row['index_in_original']
        custom_id = f"request-{idx + 1}"
        payload = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": f"{user_task}\n\n{imp}"}
                ]
            }
        }
        fout.write(json.dumps(payload) + "\n")
        # Collect mapping for this request
        impression_id_list.append(impression_id)
        custom_id_list.append(custom_id)

# Save the mapping to CSV for later correspondence
import pandas as pd
mapping_df = pd.DataFrame({"impression_id": impression_id_list, "custom_id": custom_id_list})
mapping_df.to_csv(impression_customid_mapping_filename, index=False)

print(f"Wrote {len(impressions_df)} requests to {batch_jsonl_filename} for OpenAI batch /v1/chat/completions.")
print(f"Mapping between impression_id and custom_id written to {impression_customid_mapping_filename}.")


Wrote 1207 requests to long_impressions_openai_chat_batch_3.jsonl for OpenAI batch /v1/chat/completions.
Mapping between impression_id and custom_id written to long_impressionid_customid_mapping_3.csv.


## B. Submit

In [52]:
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)

batch_input_file = client.files.create(
    file=open("long_impressions_openai_chat_batch_3.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-8u2agpA9mz2nkieWygMcbZ', bytes=878650, created_at=1763121911, filename='long_impressions_openai_chat_batch_3.jsonl', object='file', purpose='batch', status='processed', expires_at=1765713911, status_details=None)


In [53]:
from openai import OpenAI

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "summarize long impressions"
    }
)

Batch(id='batch_69171b010c188190a5fdab891ede3d26', completion_window='24h', created_at=1763121921, endpoint='/v1/chat/completions', input_file_id='file-8u2agpA9mz2nkieWygMcbZ', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1763208321, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'summarize long impressions'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [1]:
import time
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)

batch_id = "batch_69171b010c188190a5fdab891ede3d26"

while True:
    batch = client.batches.retrieve(batch_id)
    print(f"Batch: {batch}")
    print(f"Status: {batch.status}")
    print(f"Batch counts: {batch.request_counts}")
    if batch.status in ["completed", "failed", "cancelled"]:
        print("Batch processing finished.")
        break
    time.sleep(60)  # wait 1 minute before checking again

Batch: Batch(id='batch_69171b010c188190a5fdab891ede3d26', completion_window='24h', created_at=1763121921, endpoint='/v1/chat/completions', input_file_id='file-8u2agpA9mz2nkieWygMcbZ', object='batch', status='in_progress', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1763208321, failed_at=None, finalizing_at=None, in_progress_at=1763121982, metadata={'description': 'summarize long impressions'}, model='gpt-5-2025-08-07', output_file_id=None, request_counts=BatchRequestCounts(completed=983, failed=0, total=1207), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))
Status: in_progress
Batch counts: BatchRequestCounts(completed=983, failed=0, total=1207)
Batch: Batch(id='batch_69171b010c188190a5fdab891ede3d26', completion_window='24h', created_at=1763121921, endpoint='/v1/chat/comple

## C. Retreive

In [7]:
import json
import pandas as pd
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)
import json

file_response = client.files.content("file-DrL6BwKeyLqSwFVUT2gLsy")
output_path = "output_impressions_openai_batch3.jsonl"

with open(output_path, "w") as f:
    for line in file_response.text.strip().splitlines():
        # Each line should already be json-serialized; optionally validate/parsing:
        try:
            obj = json.loads(line)
            f.write(json.dumps(obj) + "\n")
        except json.JSONDecodeError:
            # If it's plain text, wrap in JSON
            f.write(json.dumps({"text": line}) + "\n")

print(f"Wrote file to {output_path}")


# Extract relevant fields from jsonl lines and build list of dicts
records = []
for line in file_response.text.strip().splitlines():
    try:
        obj = json.loads(line)
        # get id and summary content
        record = {
            "id": obj.get("id"),
            "custom_id": obj.get("custom_id"),
        }
        # Try to extract summary text (OpenAI batch response format)
        try:
            record["summarized"] = obj["response"]["body"]["choices"][0]["message"]["content"]
        except Exception:
            record["summarized"] = None
        records.append(record)
    except Exception:
        pass

summarized_batch3 = pd.DataFrame(records)


Wrote file to output_impressions_openai_batch3.jsonl


In [70]:

import json

error_response = client.files.content("file-7Tsv1kJD4WfbKBnkikxH7L")
output_path = "error_impressions_openai.jsonl"

with open(output_path, "w") as f:
    for line in error_response.text.strip().splitlines():
        # Each line should already be json-serialized; optionally validate/parsing:
        try:
            obj = json.loads(line)
            f.write(json.dumps(obj) + "\n")
        except json.JSONDecodeError:
            # If it's plain text, wrap in JSON
            f.write(json.dumps({"text": line}) + "\n")

print(f"Wrote file to {output_path}")


Wrote file to error_impressions_openai.jsonl
